# RAG Anything — Showcase Indexing, NER, Graph, Summarization, And Q&A

This notebook indexes and parses files from `./data/raw/showcase`, creates embeddings through the LightRAG-backed ingestion path, extracts entities and relations, summarizes the resulting document thesaurus, visualizes the thesaurus and a graph snapshot, and runs Q&A over the indexed collection.


## Techniques As Parameters

The workflow exposes configurable technique knobs through a single `ShowcaseTechniques` object:

- `parser`
- `parse_method`
- `query_mode`
- `response_type`
- `query_optimization_strategy`
- `context_window`
- `enable_vlm`
- `top_k`

These settings map to the currently implemented pipeline, not to hypothetical capabilities. In this repository, the most relevant state-of-the-art path is the LightRAG-backed graph + vector retrieval pipeline with multimodal processing and query optimization.


In [ ]:
from pathlib import Path
import json
import pandas as pd

from src.notebook_utils import ensure_project_root_on_path, ensure_dir

ensure_project_root_on_path()

from src.create_thesaurus import create_thesaurus
from src.showcase_rag import (
    ShowcaseTechniques,
    index_showcase_dataset_sync,
    summarize_showcase_sync,
    query_showcase_thesaurus_sync,
)
from src.viz import plot_thesaurus_distribution, plot_knowledge_graph

SHOWCASE_ROOT = ensure_dir(Path('./data/raw/showcase'))
OUTPUT_ROOT = ensure_dir(Path('./output/showcase'))
print(f'Showcase root: {SHOWCASE_ROOT}')
print(f'Output root:   {OUTPUT_ROOT}')


## Define The Pipeline Configuration


In [ ]:
techniques = ShowcaseTechniques(
    parser='mineru',
    parse_method='auto',
    query_mode='mix',
    response_type='Multiple Paragraphs',
    query_optimization_strategy='all',
    context_window=1,
    enable_vlm=False,
    top_k=10,
)

techniques


## Build The Local Thesaurus Before Indexing

This is the document inventory that the indexing step will operate on.


In [ ]:
pre_index_thesaurus = create_thesaurus(SHOWCASE_ROOT)
print(json.dumps({
    'total_files': pre_index_thesaurus['total_files'],
    'total_size_bytes': pre_index_thesaurus['total_size_bytes'],
    'groups': pre_index_thesaurus['groups'],
}, indent=2))


In [ ]:
pd.DataFrame(pre_index_thesaurus['files']).head(100) if pre_index_thesaurus['files'] else pd.DataFrame(columns=['path', 'name', 'extension', 'group', 'size_bytes'])


## Index, Parse, And Create Embeddings

The indexing step uses `process_documents_with_rag_batch(...)`, which drives parsing, text insertion into LightRAG, multimodal processing, embedding creation, and entity/relation extraction through the existing pipeline.


In [ ]:
pipeline_result = index_showcase_dataset_sync(
    input_location=str(SHOWCASE_ROOT),
    output_location=str(OUTPUT_ROOT),
    techniques=techniques,
)

print(json.dumps({
    'input_location': pipeline_result.input_location,
    'techniques': pipeline_result.techniques,
    'ingestion_keys': list(pipeline_result.ingestion.keys()),
    'graph_node_count': pipeline_result.graph_summary.get('node_count', 0),
    'graph_edge_count': pipeline_result.graph_summary.get('edge_count', 0),
}, indent=2))


## Inspect The Ingestion Result


In [ ]:
ingestion = pipeline_result.ingestion
print(json.dumps(ingestion, indent=2)[:12000])


## NER, Entity Recognition, And Entity Relation Discovery Snapshot

Entity and relation extraction are produced during ingestion. The graph snapshot below reflects what is currently stored in the LightRAG-backed entity/relation graph after indexing.


In [ ]:
graph_summary = pipeline_result.graph_summary
print(json.dumps(graph_summary, indent=2)[:12000])


In [ ]:
plot_knowledge_graph(graph_summary, title='Showcase Entity / Relation Snapshot')


## Visualize The Document Thesaurus


In [ ]:
plot_thesaurus_distribution(
    pipeline_result.thesaurus['groups'],
    title='Showcase Thesaurus Distribution',
)


## Summarize The Thesaurus With RAG


In [ ]:
summary_result = summarize_showcase_sync(
    input_location=str(SHOWCASE_ROOT),
    techniques=techniques,
)

print(summary_result['result'])


## Query / Q&A Over The Indexed Thesaurus


In [ ]:
question = 'What document types are present, what are the main entities, and what relationships can be inferred across the showcase dataset?'
qa_result = query_showcase_thesaurus_sync(
    query=question,
    input_location=str(SHOWCASE_ROOT),
    techniques=techniques,
)

print(json.dumps({
    'query': qa_result['query'],
    'selected_query': qa_result['selected_query'],
    'query_variants': qa_result['query_variants'],
    'analysis': qa_result['analysis'],
}, indent=2))


In [ ]:
print(qa_result['result'])


## Technique Variants To Try

Examples:

- `techniques.query_mode = 'hybrid'` for explicit hybrid retrieval
- `techniques.query_mode = 'global'` for more graph-centric retrieval
- `techniques.query_mode = 'local'` for chunk-local grounding
- `techniques.response_type = 'Bullet Points'` for structured summaries
- `techniques.query_optimization_strategy = 'rewrite'` or `'expand'`
- `techniques.context_window = 2` for richer multimodal context
- `techniques.enable_vlm = True` when a vision model is configured and the retrieved context includes images
